# Pass 3: Actions Demanded
LLM coding of actions called for in YouTube crisis comments.
Batch-by-batch processing per crisis event.

## Methodology

### Motivation
Emotions (Passes 1–2) explain *how* the public feels; Pass 3 captures
*what they want done* — the actionable demand layer (investigate, resign,
compensate, reform, punish, "None"/implicit).

### Approach
Same chunked protocol with `../prompts/pass-03-actions-demanded.md`.
Single string `action_demanded` per comment; "None" or implicit-demand
description when no explicit call.

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1–4** | Same as Pass 1 | Load → slim/clean → chunk (15) → LLM call + `json.loads` |
| **5. Export** | Save + inspect | `sample_pass03_results.jsonl` + `.csv` → merge on `row_id` for QA |

### Cross-Analysis
- *Which emotions (Passes 1–2) predict which demands?*
- *Do demands escalate across phases (onset → lingering)?*

### Input / Output
- **Input:** `01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv`
- **Output:** `03-llm-passes/output/{crisis_name}/pass-03-actions-demanded/sample_pass03_results.jsonl` + `.csv` (same basename)

## Setup
Import dependencies: `pandas` for data, `dotenv` + `google.genai` for LLM
calls, `json/re/time/pathlib` for chunking, parsing, and checkpointing.

In [21]:
import pandas as pd
import numpy as np
import os
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
import random
from tqdm.auto import tqdm

## Configuration
Micro-sample selector. Papermill injects `crisis_name` / `phase_name` per run.

In [22]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "04-reckoning"  # default; overridden by papermill
n_sample = 200
chunk_size = 5
sample_seed = 42
model = "gemma-4-31b-it"
overwrite = False

In [23]:
# ── Environment & API Setup ─────────────────────────────────────────────
load_dotenv("../../.env")
GEMMA_API_KEY = os.environ.get("GEMMA_API_KEY", "")

if not GEMMA_API_KEY:
    raise RuntimeError("GEMMA_API_KEY missing. Copy .env.example → .env and fill it.")
print("✅ Gemma API key loaded.")
client = genai.Client(api_key=GEMMA_API_KEY)

# ── Path Configurations ─────────────────────────────────────────────────
INPUT_PATH  = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"
PASS_DIR    = f"../output/{crisis_name}/pass-03-actions-demanded"
OUTPUT_PATH = f"{PASS_DIR}/sample_pass03_results.jsonl"
LOG_PATH    = f"{PASS_DIR}/sample_pass03_failures.log"
CSV_PATH    = OUTPUT_PATH.replace(".jsonl", ".csv")
# Pre-subfolder location (read-only fallback for existing files)
LEGACY_OUTPUT_PATH = f"../output/{crisis_name}/sample_pass03_results.jsonl"
PARTIAL_RESP = OUTPUT_PATH + ".partial.jsonl"
PARTIAL_FAIL = LOG_PATH + ".partial.log"

# ── Model & Processing Configurations ───────────────────────────────────
MODEL       = model  # "gemma-4-31b-it" via parameters
TEMPERATURE = 0.0
MAX_TOKENS  = 8192  # max output ceiling; model uses only what it needs
REQUEST_TIMEOUT_MS = 60000  # per-request cap; stalls raise into transient retry
BATCH_SIZE  = chunk_size  # 15 default from parameters
WORKERS     = 5           # concurrent threads (used when threaded runner lands)

✅ Gemma API key loaded.


## Micro-sample ingest & preprocess
Load `n_sample` rows from `INPUT_PATH`, slim to `video_id` + `text`,
drop nulls, strip whitespace. Keeps `row_id` for lossless merge at inspect time.

In [24]:
full_df = pd.read_csv(INPUT_PATH)
print(f"Full phase rows: {len(full_df)}")

sample_df = full_df.head(n_sample).copy()  # deterministic for review; n_sample=200
sample_df = sample_df[["video_id", "text"]]
sample_df = sample_df.dropna()
sample_df["text"] = sample_df["text"].astype(str).str.strip()
sample_df = sample_df[sample_df["text"] != ""].reset_index(drop=True)
sample_df["row_id"] = sample_df.index
print(f"Sample rows: {len(sample_df)}")
sample_df.head()

Full phase rows: 1411
Sample rows: 200


,video_id,text,row_id
0,dtvrM51u4Ks,an eye for an eye and tooth for tooth.stop thi...,0
1,dtvrM51u4Ks,justice for the 2 young guys and no one will g...,1
2,dtvrM51u4Ks,if he apologizes that means guilty in the deat...,2
3,dtvrM51u4Ks,you should stay here in the philippines until ...,3
4,dtvrM51u4Ks,did you go to their wake?,4


## Batch chunking
Slice the sample into 15-row chunks and convert each chunk to a raw CSV string
for one LLM call apiece.

In [25]:
chunks = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
chunk_csvs = [c[["video_id", "text"]].to_csv(index=False) for c in chunks]
print(f"{len(chunks)} chunks x ~{BATCH_SIZE} rows | first chunk chars: {len(chunk_csvs[0])}")
print(chunk_csvs[0][:600])

40 chunks x ~5 rows | first chunk chars: 448
video_id,text
dtvrM51u4Ks,"an eye for an eye and tooth for tooth.stop this scripted lies, go to hell"
dtvrM51u4Ks,justice for the 2 young guys and no one will go thru this
dtvrM51u4Ks,if he apologizes that means guilty in the death of 2 players
dtvrM51u4Ks,you should stay here in the philippines until justice given to rene and divine and to their berieved family. don't hide. face their families personally.
dtvrM51u4Ks,did you go to their wake?



## LLM inference
Pass the system instruction + one CSV chunk per call to the Google AI Studio API.
Sequential calls with a 1s pause; raw responses kept in memory for the parse step.

In [26]:
prompt_template = Path("../prompts/pass-03-actions-demanded.txt").read_text(encoding="utf-8").strip()
print(prompt_template[:400])

TRANSIENT_SLEEP = (3, 5)  # pilot; flip to (10, 10) for full run
RATE_LIMIT_SLEEP = (60, 90)

def is_rate_limited(e: Exception) -> bool:
    msg = repr(e)
    return ("429" in msg) or ("RESOURCE_EXHAUSTED" in msg)

def is_transient(e: Exception) -> bool:
    msg = repr(e)
    return any(k in msg for k in ("500", "INTERNAL", "503", "UNAVAILABLE", "HIGH DEMAND",
                                  "Timeout", "TIMEOUT", "TimeoutError", "timed out"))

def call_with_retry(full_input, tries=4):
    for attempt in range(tries):
        try:
            return client.models.generate_content(model=MODEL, contents=full_input,
                config=types.GenerateContentConfig(temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS,
                    http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_MS)))
        except Exception as e:
            if attempt == tries - 1:
                raise
            if is_rate_limited(e):
                sleep_time = random.uniform(*RATE_LIMIT_SLEEP)
            elif is_transient(e):
                sleep_time = random.uniform(*TRANSIENT_SLEEP)
            else:
                sleep_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"  [Attempt {attempt+1} failed: {repr(e)}] Retrying in {sleep_time:.2f}s...")
            time.sleep(sleep_time)

chunks_queue = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
raw_responses, hard_failures = [], []
# Resume-skip: reload succeeded row_ids from partials when not overwriting
# Prior api_failures are retried (not skipped) — only ok_ids filter the queue.
if not overwrite:
    try:
        ok_ids = set()
        if Path(PARTIAL_RESP).exists():
            with open(PARTIAL_RESP, encoding="utf-8") as f:
                for line in f:
                    try:
                        r = json.loads(line)
                        ok_ids.update(r.get("row_ids", []))
                        raw_responses.append(r)
                    except Exception:
                        pass
        prior_failures = []
        if Path(PARTIAL_FAIL).exists():
            with open(PARTIAL_FAIL, encoding="utf-8") as f:
                for line in f:
                    try:
                        e = json.loads(line)
                        if "row_id" in e:
                            prior_failures.append(e)
                    except Exception:
                        pass
            # Clear stale failure checkpoint so a retry appends cleanly instead of duplicating.
            try:
                Path(PARTIAL_FAIL).unlink()
            except Exception:
                pass
        hard_failures = []
        if ok_ids:
            chunks_queue = [c for c in chunks_queue if not set(c["row_id"].tolist()) <= ok_ids]
            print(f"Resuming: {len(ok_ids)} rows done, {len(prior_failures)} prior failures to retry, {len(chunks_queue)} chunks left.")
    except Exception as e:
        print(f"Resume check failed ({e!r}); starting fresh.")
else:
    for p in (PARTIAL_RESP, PARTIAL_FAIL):
        try:
            Path(p).unlink(missing_ok=True)
        except Exception:
            pass
print(f"Starting dynamic queue with {len(chunks_queue)} initial chunks...")

# Row-level progress (split-safe: total fixed, update on terminal rows only)
total_rows = len(sample_df)
_resumed = len(ok_ids) if (not overwrite and 'ok_ids' in locals()) else 0
pbar = tqdm(total=total_rows, initial=_resumed,
            desc=f"LLM {crisis_name}/{phase_name}", unit="rows")

try:
    while chunks_queue:
        current_chunk = chunks_queue.pop(0)
        n_rows = len(current_chunk)
        csv_str = current_chunk[["video_id", "text"]].to_csv(index=False)
        try:
            resp = call_with_retry(prompt_template + "\n\nData:\n" + csv_str)
            rec = {"row_ids": current_chunk["row_id"].tolist(), "text": resp.text or ""}
            raw_responses.append(rec)
            with open(PARTIAL_RESP, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            pbar.update(n_rows)
            pbar.set_postfix(ok=len(raw_responses), fail=len(hard_failures), left=len(chunks_queue))
            time.sleep(1)
        except Exception as e:
            print(f"❌ Chunk failure (Size: {n_rows}): {repr(e)}")
            if is_rate_limited(e) or is_transient(e):
                print(f"   ↳ Transient exhausted after retries. Logging {n_rows} rows and skipping (no split).")
                for _, row in current_chunk.iterrows():
                    ent = {"row_id": int(row["row_id"]), "video_id": row.get("video_id"),
                           "reason": "transient_exhausted", "error": repr(e),
                           "text_length": len(str(row["text"]))}
                    hard_failures.append(ent)
                    with open(PARTIAL_FAIL, "a", encoding="utf-8") as f:
                        f.write(json.dumps({"kind": "api_failure", **ent}, ensure_ascii=False) + "\n")
                pbar.update(n_rows)
                pbar.set_postfix(ok=len(raw_responses), fail=len(hard_failures), left=len(chunks_queue))
                continue
            if n_rows > 1:
                print(f"   ↳ Splitting chunk of {n_rows}...")
                mid = n_rows // 2
                chunks_queue.insert(0, current_chunk.iloc[mid:])
                chunks_queue.insert(0, current_chunk.iloc[:mid])
                # no pbar.update — rows requeued, counted on terminal resolution
                pbar.set_postfix(ok=len(raw_responses), fail=len(hard_failures), left=len(chunks_queue))
            else:
                row = current_chunk.iloc[0]
                print(f"   ↳ Hard failure on row_id {row['row_id']}. Logging and skipping.")
                ent = {"row_id": int(row["row_id"]), "video_id": row.get("video_id"),
                       "reason": "content_failure", "error": repr(e), "text_length": len(str(row["text"]))}
                hard_failures.append(ent)
                with open(PARTIAL_FAIL, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"kind": "api_failure", **ent}, ensure_ascii=False) + "\n")
                pbar.update(1)
                pbar.set_postfix(ok=len(raw_responses), fail=len(hard_failures), left=len(chunks_queue))
except (KeyboardInterrupt, SystemExit):
    pbar.close()
    print(f"\nInterrupted: {len(raw_responses)} chunks done, {len(chunks_queue)} left. "
          f"Partials at {PARTIAL_RESP} / {PARTIAL_FAIL}.")
    raise

pbar.close()
print(f"\nDone. {len(raw_responses)} ok chunks, {len(hard_failures)} hard failures.")

System Instruction:
I will provide a dataset of comments in CSV format containing `video_id` and `text` columns. You must return the analysis strictly as a JSON array of objects. Do not include markdown formatting (other than the JSON block) or explanations outside of the JSON.

For each comment's text, identify the action that is being called or demanded.

Use this exact JSON structure for each o
Resuming: 116 rows done, 94 prior failures to retry, 19 chunks left.
Starting dynamic queue with 19 initial chunks...


LLM ateneo-drowning/04-reckoning:  58%|█████▊    | 116/200 [00:00<?, ?rows/s]

  [Attempt 1 failed: ServerError("504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired before operation could complete.', 'status': 'DEADLINE_EXCEEDED'}}")] Retrying in 1.38s...
  [Attempt 2 failed: ServerError("504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired before operation could complete.', 'status': 'DEADLINE_EXCEEDED'}}")] Retrying in 2.41s...
  [Attempt 3 failed: ServerError("500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}")] Retrying in 3.94s...


LLM ateneo-drowning/04-reckoning:  58%|█████▊    | 116/200 [03:39<?, ?rows/s, fail=0, left=20, ok=32]

❌ Chunk failure (Size: 5): ServerError("504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired before operation could complete.', 'status': 'DEADLINE_EXCEEDED'}}")
   ↳ Splitting chunk of 5...
  [Attempt 1 failed: ServerError("500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}")] Retrying in 3.57s...
  [Attempt 2 failed: ServerError("500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}")] Retrying in 3.79s...


LLM ateneo-drowning/04-reckoning:  59%|█████▉    | 118/200 [04:56<3:22:26, 148.13s/rows, fail=0, left=19, ok=33]

... (4 outputs left out)

❌ Chunk failure (Size: 5): ServerError("504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired before operation could complete.', 'status': 'DEADLINE_EXCEEDED'}}")
   ↳ Splitting chunk of 5...
  [Attempt 1 failed: ServerError("500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}")] Retrying in 3.30s...
  [Attempt 2 failed: ServerError("500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}")] Retrying in 3.18s...
  [Attempt 3 failed: ServerError("500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}")] Retrying in 4.65s...


LLM ateneo-drowning/04-reckoning:  60%|██████    | 121/200 [08:20<2:11:54, 100.18s/rows, fail=0, left=19, ok=34]


Interrupted: 34 chunks done, 18 left. Partials at ../output/ateneo-drowning/pass-03-actions-demanded/sample_pass03_results.jsonl.partial.jsonl / ../output/ateneo-drowning/pass-03-actions-demanded/sample_pass03_failures.log.partial.log.


KeyboardInterrupt: 

## Clean & parse
Strip markdown fences from each raw response, `json.loads()` it, and map parsed
objects back to exact sample rows via the queued `row_ids`. API `hard_failures`
and unparseable chunks both flush to `LOG_PATH`.

In [27]:
# --- STANDALONE-BOOTSTRAP (no-op in a full top-to-bottom run) ---
# Lets this cell run in a fresh kernel without re-calling the LLM.
import json
import re
from pathlib import Path
crisis_name = globals().get("crisis_name", "ateneo-drowning")
phase_name = globals().get("phase_name", "04-reckoning")
n_sample = globals().get("n_sample", 200)
if "OUTPUT_PATH" not in globals() or "LOG_PATH" not in globals() or "PARTIAL_RESP" not in globals():
    PASS_DIR = f"../output/{crisis_name}/pass-03-actions-demanded"
    OUTPUT_PATH = f"{PASS_DIR}/sample_pass03_results.jsonl"
    LOG_PATH = f"{PASS_DIR}/sample_pass03_failures.log"
    PARTIAL_RESP = OUTPUT_PATH + ".partial.jsonl"
    PARTIAL_FAIL = LOG_PATH + ".partial.log"
hard_failures = globals().get("hard_failures", [])
if "raw_responses" not in globals() or not raw_responses:
    _partial = Path(PARTIAL_RESP)
    if not _partial.exists():
        raise RuntimeError(
            f"No raw_responses in memory and no partial file at {_partial}. "
            "Run the LLM inference cells first (top-to-bottom)."
        )
    raw_responses = [
        json.loads(ln) for ln in _partial.read_text(encoding="utf-8").splitlines() if ln.strip()
    ]
    if not raw_responses:
        raise RuntimeError(
            f"Partial file at {_partial} exists but yielded 0 rows. "
            "Run the LLM inference cells first (top-to-bottom)."
        )
    print(f"Reloaded {len(raw_responses)} chunks from {_partial} (fresh-kernel resume, no LLM call).")
Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
_total_rows = (
    len(sample_df) if "sample_df" in globals() and sample_df is not None
    else sum(len(item.get("row_ids", [])) for item in raw_responses)
)

def strip_fences(raw: str) -> str:
    s = raw.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)
    return s.strip()

REQUIRED_KEYS = {"video_id", "text", "action_demanded"}

parsed_records, parse_failures = [], []
for item in raw_responses:
    row_ids, txt = item["row_ids"], item["text"]
    if not txt.strip():
        parse_failures.append({"row_ids": row_ids, "error": "empty_response"})
        continue
    try:
        objs = json.loads(strip_fences(txt))
    except Exception as e:
        parse_failures.append({"row_ids": row_ids, "error": f"json_decode: {e!r}"})
        continue
    if isinstance(objs, dict):
        objs = [objs]
    if len(objs) != len(row_ids):
        parse_failures.append({"row_ids": row_ids, "error": f"count_mismatch: {len(objs)} objs for {len(row_ids)} rows"})
    for rid, o in zip(row_ids, objs):
        missing = REQUIRED_KEYS - set(o.keys())
        if missing:
            parse_failures.append({"row_ids": [rid], "error": f"missing_keys: {sorted(missing)}"})
            continue
        o["_row_id"] = rid
        parsed_records.append(o)

with open(LOG_PATH, "w", encoding="utf-8") as f:
    for x in hard_failures:
        f.write(json.dumps({"kind": "api_failure", **x}, ensure_ascii=False) + "\n")
    for x in parse_failures:
        f.write(json.dumps({"kind": "parse_failure", **x}, ensure_ascii=False) + "\n")

print(f"parsed: {len(parsed_records)}/{_total_rows} rows | parse failures: {len(parse_failures)} | api failures: {len(hard_failures)}")

parsed: 123/200 rows | parse failures: 0 | api failures: 0


## Inspect & save
Write parsed records to `OUTPUT_PATH` as JSONL, reload it, and merge back to the
sample for visual QA. Merge key is `_row_id == row_id` (exact, split-safe) —
not `video_id`, which repeats across comments and would fan out the join.

In [28]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    for r in parsed_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Wrote {len(parsed_records)} records → {out_path}")

res_df = pd.read_json(out_path, lines=True)
merged = sample_df.merge(res_df, left_on="row_id", right_on="_row_id", how="left", suffixes=("", "_llm"))
coverage = merged["_row_id"].notna().mean()
print(f"Coverage: {coverage:.1%} ({merged['_row_id'].notna().sum()}/{len(merged)} rows matched)")
display(merged[["row_id", "video_id", "text", "action_demanded"]].head(10))
print(res_df["action_demanded"].astype(str).value_counts().to_string())

Wrote 123 records → ../output/ateneo-drowning/pass-03-actions-demanded/sample_pass03_results.jsonl
Coverage: 60.0% (123/205 rows matched)


,row_id,video_id,text,action_demanded
0,0,dtvrM51u4Ks,an eye for an eye and tooth for tooth.stop thi...,Stop the scripted lies
1,1,dtvrM51u4Ks,justice for the 2 young guys and no one will g...,Justice for the two young guys
2,2,dtvrM51u4Ks,if he apologizes that means guilty in the deat...,None
3,2,dtvrM51u4Ks,if he apologizes that means guilty in the deat...,None
4,3,dtvrM51u4Ks,you should stay here in the philippines until ...,Stay in the Philippines until justice is serve...
5,3,dtvrM51u4Ks,you should stay here in the philippines until ...,Stay in the Philippines until justice is serve...
6,4,dtvrM51u4Ks,did you go to their wake?,None
7,4,dtvrM51u4Ks,did you go to their wake?,None
8,5,dtvrM51u4Ks,si coach tab ang nag papatunay na basta kalbo ...,NaN
9,6,dtvrM51u4Ks,"surrender yourself and admit your mistake, go ...",NaN


action_demanded
None                                                                                                                         52
Go to jail                                                                                                                    5
Imprisonment                                                                                                                  3
Stay in the Philippines until justice is served, do not hide, and face the families personally.                               2
Be held accountable and face charges.                                                                                         1
Inform students and parents about the potential risk of death during training.                                                1
Empathize with the victim's mother.                                                                                           1
Resign, face the investigation, and allow a new coach to take over.                     

## Summary counts (in-memory, no LLM call)
QA gate before CSV export: row count plus distributions of the coded
fields. Uses the in-memory frames from Inspect & save when present;
falls back to `OUTPUT_PATH` on disk for standalone reruns.

In [29]:
# ── Summary counts: rows + coded-field distributions ──
# Prefers in-memory frames (top-to-bottom run); falls back to OUTPUT_PATH on disk.
# Coded columns auto-detected (everything except join/text keys) so one cell fits any pass.
import pandas as _pd_sum
_SKIP_SUM = {"row_id", "video_id", "text", "_row_id", "row_ids"}
_tbl = None
if "merged" in globals() and merged is not None and len(merged):
    _tbl = merged
elif "res_df" in globals() and res_df is not None and len(res_df):
    _tbl = res_df
else:
    try:
        _tbl = _pd_sum.read_json(OUTPUT_PATH, lines=True)
        print(f"(standalone fallback — read {len(_tbl)} records from {OUTPUT_PATH})")
    except Exception as _e:
        print(f"⚠️ No in-memory frames and cannot read {OUTPUT_PATH}: {_e!r}")
        print("Run the Clean & parse + Inspect cells first.")
        _tbl = None
if _tbl is not None:
    # Row count + coverage when the merged join key exists
    if "_row_id" in _tbl.columns:
        _n_ok = int(_tbl["_row_id"].notna().sum())
        print(f"n rows: {len(_tbl)} (matched {_n_ok}, missing {len(_tbl) - _n_ok})")
    else:
        print(f"n rows: {len(_tbl)}")
    # Coded-field summaries; list cols stringified so CSV and memory forms agree
    for _c in [c for c in _tbl.columns if c not in _SKIP_SUM]:
        _s = _tbl[_c].astype(str) if _tbl[_c].apply(lambda v: isinstance(v, list)).any() else _tbl[_c]
        print(f"\n== {_c} ==")
        print(_s.value_counts(dropna=False).head(20).to_string())


n rows: 205 (matched 123, missing 82)

== video_id_llm ==
video_id_llm
NaN            82
Q6dGkiMTes0    65
dtvrM51u4Ks    43
ifzyUElx76Y    10
OXk-mjXkQYs     5

== text_llm ==
text_llm
NaN                                                                                                                                                                                                                                                                          82
if he apologizes that means guilty in the death of 2 players                                                                                                                                                                                                                  2
you should stay here in the philippines until justice given to rene and divine and to their berieved family. don't hide. face their families personally.                                                                                                                      

## Export CSV from disk (rerunnable, no LLM call)
Reads the saved JSONL (`OUTPUT_PATH`, else `*.partial.jsonl`, else legacy flat path)
and writes the merged QA table to `CSV_PATH`. Handles both final parsed records
(`_row_id` present) and raw chunk partials (`row_ids` + fenced `text`).
Safe to run standalone right after configuring paths — no API key needed.

In [30]:
# ── Rerunnable CSV export (reads JSONL from disk; no LLM call) ──────────
# Standalone: safe to run in a fresh kernel without running setup/config cells.
import json
import json as _json
import re
from pathlib import Path
import pandas as pd

# Bootstrap defaults when run standalone (no-op in a full top-to-bottom run)
crisis_name = globals().get("crisis_name", "ateneo-drowning")
phase_name = globals().get("phase_name", "04-reckoning")
n_sample = globals().get("n_sample", 200)
if "OUTPUT_PATH" not in globals():
    PASS_DIR = f"../output/{crisis_name}/pass-03-actions-demanded"
    OUTPUT_PATH = f"{PASS_DIR}/sample_pass03_results.jsonl"
    LOG_PATH = f"{PASS_DIR}/sample_pass03_failures.log"
    CSV_PATH = OUTPUT_PATH.replace(".jsonl", ".csv")
    LEGACY_OUTPUT_PATH = f"../output/{crisis_name}/sample_pass03_results.jsonl"
    PARTIAL_RESP = OUTPUT_PATH + ".partial.jsonl"
    INPUT_PATH = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"


_candidates = [OUTPUT_PATH, PARTIAL_RESP, LEGACY_OUTPUT_PATH, LEGACY_OUTPUT_PATH + ".partial.jsonl"]
SRC_JSONL = next((c for c in _candidates if Path(c).exists()), None)
if SRC_JSONL is None:
    print("⚠️ No JSONL found. Checked:")
    [print(f"   - {c}") for c in _candidates]
    print("Run the LLM inference + Clean & parse cells first.")
else:
    print(f"Reading: {SRC_JSONL}")
    # Rebuild sample deterministically from disk (independent of in-memory sample_df)
    _full = pd.read_csv(INPUT_PATH)
    _sample = _full.head(n_sample)[["video_id", "text"]].copy()
    _sample = _sample.dropna()
    _sample["text"] = _sample["text"].astype(str).str.strip()
    _sample = _sample[_sample["text"] != ""].reset_index(drop=True)
    _sample["row_id"] = _sample.index

    # Load lines; distinguish parsed records vs raw chunk partials
    _raw_lines = [json.loads(ln) for ln in open(SRC_JSONL, encoding="utf-8") if ln.strip()]
    if _raw_lines and "row_ids" in _raw_lines[0]:
        # Raw chunks → parse exactly like the Clean & parse cell
        def _strip_fences(raw: str) -> str:
            s = raw.strip()
            s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
            s = re.sub(r"\s*```$", "", s)
            return s.strip()
        _REQUIRED = {"video_id", "text", "action_demanded"}
        _parsed, _n_bad = [], 0
        for item in _raw_lines:
            _rids, _txt = item.get("row_ids", []), item.get("text", "")
            if not str(_txt).strip():
                _n_bad += 1
                continue
            try:
                _objs = json.loads(_strip_fences(_txt))
            except Exception:
                _n_bad += 1
                continue
            if isinstance(_objs, dict):
                _objs = [_objs]
            for _rid, _o in zip(_rids, _objs):
                if _REQUIRED - set(_o.keys()):
                    _n_bad += 1
                    continue
                _o["_row_id"] = _rid
                _parsed.append(_o)
        print(f"Parsed {len(_parsed)} records from raw chunks (skipped {_n_bad} bad).")
        res_df = pd.DataFrame(_parsed)
    else:
        res_df = pd.read_json(SRC_JSONL, lines=True)

    merged_disk = _sample.merge(res_df, left_on="row_id", right_on="_row_id", how="left", suffixes=("", "_llm"))
    for _c in ("video_id_llm", "text_llm"):
        if _c in merged_disk.columns:
            merged_disk = merged_disk.drop(columns=[_c])
    # Serialize list cols so CSV round-trips via json.loads
    for _lc in []:
        if _lc in merged_disk.columns:
            merged_disk[_lc] = merged_disk[_lc].apply(lambda v: _json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v)
    keep = [c for c in ["row_id", "video_id", "text", "action_demanded"] if c in merged_disk.columns]
    out_csv = Path(CSV_PATH)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    merged_disk.sort_values("row_id")[keep].to_csv(out_csv, index=False)
    _cov = merged_disk["_row_id"].notna().mean() if "_row_id" in merged_disk.columns else 0.0
    print(f"Wrote {len(merged_disk)} rows (coverage {_cov:.1%}) → {out_csv}")
    try:
        display(merged_disk[keep].head(10))
    except NameError:
        print(merged_disk[keep].head(10).to_string())

Reading: ../output/ateneo-drowning/pass-03-actions-demanded/sample_pass03_results.jsonl
Wrote 205 rows (coverage 60.0%) → ../output/ateneo-drowning/pass-03-actions-demanded/sample_pass03_results.csv


,row_id,video_id,text,action_demanded
0,0,dtvrM51u4Ks,an eye for an eye and tooth for tooth.stop thi...,Stop the scripted lies
1,1,dtvrM51u4Ks,justice for the 2 young guys and no one will g...,Justice for the two young guys
2,2,dtvrM51u4Ks,if he apologizes that means guilty in the deat...,None
3,2,dtvrM51u4Ks,if he apologizes that means guilty in the deat...,None
4,3,dtvrM51u4Ks,you should stay here in the philippines until ...,Stay in the Philippines until justice is serve...
5,3,dtvrM51u4Ks,you should stay here in the philippines until ...,Stay in the Philippines until justice is serve...
6,4,dtvrM51u4Ks,did you go to their wake?,None
7,4,dtvrM51u4Ks,did you go to their wake?,None
8,5,dtvrM51u4Ks,si coach tab ang nag papatunay na basta kalbo ...,NaN
9,6,dtvrM51u4Ks,"surrender yourself and admit your mistake, go ...",NaN
